# 11.05 · Production feedback loops

**Chapter:** LLMOps, Observability & Safety · **Milestone:** M5

## Goals
- [ ] Capture explicit (👍/👎, edits) and implicit (retries, abandonment, copy) feedback, linked to traces
- [ ] Online evaluation: run cheap checks on 100% of traffic, LLM judges on a sample
- [ ] Close the loop: failing production traces become new eval cases
- [ ] A/B test prompt/model variants and read the result honestly
- [ ] Detect drift in what users ask

## Resources
- [Chip Huyen – AI Engineering (book), ch. 10 "AI Engineering Architecture and User Feedback"](https://www.oreilly.com/library/view/ai-engineering/9781098166298/) — **paid**, recommended
- [Langfuse – Scores & user feedback](https://langfuse.com/docs/scores/user-feedback) · [Phoenix – Online evals](https://arize.com/docs/phoenix) — free
- [Eugene Yan – Patterns for building LLM-based systems](https://eugeneyan.com/writing/llm-patterns/) — free

The flywheel: **ship → observe (traces + feedback) → find failures → add to eval set → fix → verify with evals → ship**.

In [ ]:
import hashlib
import json
import random
import sqlite3
import uuid

import numpy as np

from aieng import DATA, rag

## 1. Store interactions and feedback together
Every response gets an id (= trace id, 11.01). Feedback references it. Here: one SQLite DB simulating a production log.

In [ ]:
db_path = DATA / "prod.sqlite"
db_path.unlink(missing_ok=True)
db = sqlite3.connect(db_path)
db.executescript("""
CREATE TABLE interactions (id TEXT PRIMARY KEY, ts INTEGER, user_id TEXT, variant TEXT, question TEXT, answer TEXT, retrieved TEXT, latency_ms INTEGER);
CREATE TABLE feedback (interaction_id TEXT REFERENCES interactions(id), kind TEXT, value INTEGER, comment TEXT);
""")

## 2. A/B assignment
Assign variants **deterministically** by user (hash), so a user always sees the same variant. Variant B here: retrieve 8 passages instead of 3.

In [ ]:
VARIANTS = {"A": {"k": 3}, "B": {"k": 8}}


def variant_for(user_id):
    return "A" if int(hashlib.sha256(user_id.encode()).hexdigest(), 16) % 2 == 0 else "B"


print({u: variant_for(u) for u in ["ana", "luis", "marta", "jorge"]})

## 3. Simulate traffic
Real users would click 👍/👎. We simulate them with a rule: thumbs-up iff the gold answer string is in the answer (plus noise — real feedback is
noisy and sparse: most users never click anything).

In [ ]:
rng = random.Random(0)
evalset = rag.eval_set()
users = [f"user{i}" for i in range(40)]

for day in range(5):
    for row in rng.sample(evalset, 30):
        user = rng.choice(users)
        v = variant_for(user)
        answer, hits = rag.answer(row["question"], k=VARIANTS[v]["k"], max_tokens=200)
        iid = str(uuid.uuid4())
        db.execute("INSERT INTO interactions VALUES (?,?,?,?,?,?,?,?)",
                   (iid, day, user, v, row["question"], answer, json.dumps([p for p, _, _ in hits]), rng.randint(400, 2500)))
        if rng.random() < 0.6:  # 60% of users leave feedback (generous!)
            good = row["answer"].lower().rstrip(".") in answer.lower()
            if rng.random() < 0.1:
                good = not good  # noise
            db.execute("INSERT INTO feedback VALUES (?,?,?,?)", (iid, "thumbs", int(good), None))
db.commit()
print(db.execute("SELECT COUNT(*), (SELECT COUNT(*) FROM feedback) FROM interactions").fetchone(), "(interactions, feedback)")

## 4. Read the A/B result — with a confidence interval
Thumbs-up rate per variant, plus a bootstrap CI on the difference (02.03). If the interval includes 0, you haven't shown anything.

In [ ]:
rows = db.execute("SELECT i.variant, f.value FROM feedback f JOIN interactions i ON i.id = f.interaction_id").fetchall()
by = {v: np.array([val for var, val in rows if var == v]) for v in VARIANTS}
for v, arr in by.items():
    print(f"variant {v} (k={VARIANTS[v]['k']}): {arr.mean():.1%} 👍 over {len(arr)} ratings")

boot = np.random.default_rng(0)
diffs = [boot.choice(by["B"], len(by["B"])).mean() - boot.choice(by["A"], len(by["A"])).mean() for _ in range(5000)]
lo, hi = np.percentile(diffs, [2.5, 97.5])
print(f"B - A = {by['B'].mean() - by['A'].mean():+.1%}, 95% CI [{lo:+.1%}, {hi:+.1%}] -> {'significant' if lo > 0 or hi < 0 else 'not significant: keep collecting'}")

Also compare **guardrail metrics** (latency, cost, error rate): variant B sends ~2.7× more context tokens per request.

## 5. Online evaluation
Cheap code checks on everything; an LLM judge on a sample. Store results as scores on the interaction (Langfuse/Phoenix do this natively).

In [ ]:
import re

checks = []
for iid, answer, retrieved in db.execute("SELECT id, answer, retrieved FROM interactions").fetchall():
    cited = {int(x) for x in re.findall(r"\[(\d+)\]", answer)}
    checks.append({"has_citation": bool(cited), "citations_valid": cited <= set(json.loads(retrieved)), "abstained": "don't know" in answer.lower()})
print({k: f"{np.mean([c[k] for c in checks]):.0%}" for k in checks[0]})

## 6. Close the loop: 👎 → eval cases
Pull negative-feedback interactions, review them (humans!), and append confirmed failures to the eval set (10.02) so the fix is verified and the
bug never silently returns.

In [ ]:
negatives = db.execute("""SELECT i.question, i.answer FROM feedback f JOIN interactions i ON i.id = f.interaction_id
                          WHERE f.value = 0 ORDER BY i.ts DESC LIMIT 10""").fetchall()
review_queue = DATA / "eval-from-prod-review.jsonl"
review_queue.write_text("\n".join(json.dumps({"question": q, "bad_answer": a, "expected": "", "reviewed": False}) for q, a in negatives))
print(f"{len(negatives)} negatives queued for review -> {review_queue.name}")
for q, a in negatives[:3]:
    print(f"  Q: {q}\n  A: {a[:120]!r}")

## 7. Drift: are users asking different things now?
Embed questions per time window and compare to the baseline: centroid shift and the share of questions far from anything seen before.
A sudden topic (new product, outage, news event) shows up here before it shows up in complaints.

In [ ]:
q_by_day = {d: [q for (q,) in db.execute("SELECT question FROM interactions WHERE ts = ?", (d,))] for d in range(5)}
q_by_day[5] = ["How do I reset my password?", "Why was I charged twice?", "Is the app down?", "Can't log in since the update",
               "Refund for duplicate charge", "Login page shows error 500"] * 5  # a new, unseen topic appears

base = rag.embed(q_by_day[0] + q_by_day[1])
centroid = base.mean(0) / np.linalg.norm(base.mean(0))
for d in range(2, 6):
    E = rag.embed(q_by_day[d])
    shift = 1 - float((E.mean(0) / np.linalg.norm(E.mean(0))) @ centroid)
    novel = np.mean((E @ base.T).max(1) < 0.6)
    print(f"day {d}: centroid shift {shift:.3f} | novel questions {novel:.0%}{'  <-- drift!' if shift > 0.1 else ''}")

Calibrate alert thresholds on normal days first: this corpus covers many topics, so even ordinary days have many "novel" questions; the centroid
shift is the stable signal here (~0.03 on normal days vs ~0.3 when the new topic appears).

## Exercises
1. Add an **implicit** signal: treat "user asked a near-duplicate question within 2 minutes" as a soft 👎. How well does it agree with explicit feedback?
2. Run the A/B simulation for 20 days instead of 5. When does the CI stop including 0? What's the minimum sample size for a 5-point difference?
3. Cluster the novel questions from day 5 (02.05) and auto-generate a one-line label per cluster with an LLM — a "what's new this week" report.

In [ ]:
# your code here